# Scoring
In this notebook, we score the BART-filtered headlines for "jitteriness".

First we load the labeled headlines and select only those labeled "relevant" by BART.

In [1]:
import pandas as pd

df = pd.read_csv("data/labeled_headlines.csv")
df = df[df.relevant == 1]
df.dropna(inplace=True)

Then we load the model and define the jitteriness category:

In [2]:
from transformers import pipeline
import torch

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0,
    dtype=torch.float16,
)

jittery = "jittery"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Finally, we score each headline

In [3]:
from datasets import Dataset

ds = Dataset.from_pandas(df[["concat"]])
with torch.inference_mode():
    ds = ds.map(
        lambda x: {
            "score": [
                y["scores"][0]
                for y in classifier(x["concat"], candidate_labels=jittery)
            ]
        },
        batched=True,
        batch_size=128,
    )

df["score"] = ds["score"]

Map:   0%|          | 0/973 [00:00<?, ? examples/s]

and save the scored dataset

In [6]:
df.to_csv("data/scored_headlines.csv", index=False)